# 🍳 GPT-2 Recipe Pre-training from Scratch

**Build and pre-train a GPT-2 Mini language model (~50M parameters) on a custom recipe dataset**

---

## 📋 Overview

This notebook trains a GPT-2 language model from scratch on a recipe corpus to generate coherent recipe text.

**Pipeline:**
1. Train custom BPE tokenizer (30K vocabulary)
2. Initialize GPT-2 Mini architecture (6 layers, 512 dim, 8 heads)
3. Pre-train with FP16 mixed precision for 10 epochs
4. Generate recipe text from prompts

---

## ⚙️ Setup Instructions (Google Colab Pro)

1. **Enable A100 GPU**: `Runtime` → `Change runtime type` → `A100 GPU`
2. **Upload your recipe file**: Click folder icon → Upload `recipes.txt`
3. **Update `RECIPE_FILE_PATH`** in Section 0.0 below
4. **Run all cells**: `Runtime` → `Run all`

**Estimated Time**: ~2 hours for complete training on 8,500 recipes

---

## 📁 Outputs

- `outputs/tokenizer/` — Trained BPE tokenizer (vocab.json, merges.txt)
- `outputs/gpt2-recipe-checkpoints/` — Model checkpoints (epoch 1-10)

---

# SECTION 0: USER INPUTS & HYPERPARAMETERS

> ⚠️ **CONFIGURE THESE BEFORE RUNNING THE NOTEBOOK**

In [ ]:
# ============================================================================
# SECTION 0.0: USER INPUTS (MODIFY THESE BEFORE RUNNING)
# ============================================================================
"""
📁 Path to your recipe dataset file.
   Upload to Colab runtime first, then set the path here.
   Format: Plain text file, one recipe per line with [BOS]/[EOS] markers.
"""

RECIPE_FILE_PATH = "recipes.txt"  # ⚠️ INPUT REQUIRED: Set your file path here

In [ ]:
# ============================================================================
# SECTION 0.1: TOKENIZER HYPERPARAMETERS
# ============================================================================
"""
Configuration for training the Byte Pair Encoding (BPE) tokenizer.
These settings determine vocabulary size and special token handling.
"""

TOKENIZER_CONFIG = {
    "vocab_size": 30_000,           # Target vocabulary size for BPE
    "min_frequency": 2,             # Minimum token frequency to include
    "special_tokens": [
        "[PAD]",                    # Padding token (ID: 0)
        "[UNK]",                    # Unknown token (ID: 1)
        "[BOS]",                    # Beginning of sequence (ID: 2)
        "[EOS]",                    # End of sequence (ID: 3)
    ],
}

print("✓ Tokenizer config loaded")
print(f"  Vocabulary size: {TOKENIZER_CONFIG['vocab_size']:,}")
print(f"  Special tokens: {TOKENIZER_CONFIG['special_tokens']}")

In [ ]:
# ============================================================================
# SECTION 0.2: MODEL ARCHITECTURE HYPERPARAMETERS (GPT-2 Mini)
# ============================================================================
"""
GPT-2 Mini configuration: ~50M parameters.
Optimized for training from scratch on domain-specific data.
"""

MODEL_CONFIG = {
    "vocab_size": 30_000,           # Must match tokenizer vocab_size
    "n_positions": 3000,            # Maximum sequence length (context window)
    "n_embd": 512,                  # Embedding dimension
    "n_layer": 6,                   # Number of transformer layers
    "n_head": 8,                    # Number of attention heads (must divide n_embd)
    "activation_function": "gelu_new",
    "resid_pdrop": 0.1,             # Residual dropout
    "embd_pdrop": 0.1,              # Embedding dropout
    "attn_pdrop": 0.1,              # Attention dropout
}

# Calculate approximate parameter count
approx_params = (
    MODEL_CONFIG["vocab_size"] * MODEL_CONFIG["n_embd"] +  # Token embeddings
    MODEL_CONFIG["n_positions"] * MODEL_CONFIG["n_embd"] +  # Position embeddings
    MODEL_CONFIG["n_layer"] * (
        4 * MODEL_CONFIG["n_embd"] ** 2 +  # Attention (Q, K, V, O)
        8 * MODEL_CONFIG["n_embd"] ** 2    # FFN (up + down projection)
    )
)

print("✓ Model config loaded (GPT-2 Mini)")
print(f"  Layers: {MODEL_CONFIG['n_layer']}, Embedding: {MODEL_CONFIG['n_embd']}, Heads: {MODEL_CONFIG['n_head']}")
print(f"  Max sequence length: {MODEL_CONFIG['n_positions']:,} tokens")
print(f"  Approximate parameters: ~{approx_params / 1e6:.1f}M")

In [ ]:
# ============================================================================
# SECTION 0.3: TRAINING HYPERPARAMETERS
# ============================================================================
"""
Training configuration optimized for Google Colab Pro A100 GPU (40GB VRAM).
Effective batch size = per_device_train_batch_size × gradient_accumulation_steps = 16
"""

TRAINING_CONFIG = {
    "num_train_epochs": 10,                    # Total training epochs
    "per_device_train_batch_size": 4,          # Batch size per GPU (A100 40GB allows larger batches)
    "gradient_accumulation_steps": 4,          # Effective batch size = 4 * 4 = 16
    "learning_rate": 5e-5,                     # Peak learning rate
    "weight_decay": 0.01,                      # L2 regularization
    "warmup_steps": 500,                       # Linear warmup steps
    "fp16": True,                              # Mixed precision training
    "logging_steps": 100,                      # Log every N steps
    "save_strategy": "epoch",                  # Save checkpoint every epoch
    "save_total_limit": 10,                    # Keep all 10 epoch checkpoints
    "output_dir": "./gpt2-recipe-checkpoints", # Checkpoint directory
    "report_to": "none",                       # Disable wandb/tensorboard
    "dataloader_num_workers": 2,               # Data loading workers
}

print("✓ Training config loaded")
print(f"  Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"  Batch size: {TRAINING_CONFIG['per_device_train_batch_size']} × {TRAINING_CONFIG['gradient_accumulation_steps']} = {TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']} effective")
print(f"  Learning rate: {TRAINING_CONFIG['learning_rate']}")
print(f"  FP16: {TRAINING_CONFIG['fp16']}")

In [ ]:
# ============================================================================
# SECTION 0.4: INFERENCE HYPERPARAMETERS
# ============================================================================
"""
Generation settings for recipe text inference.
These control the diversity and quality of generated text.
"""

INFERENCE_CONFIG = {
    "max_new_tokens": 200,          # Maximum tokens to generate
    "temperature": 0.8,             # Sampling temperature (higher = more random)
    "top_k": 50,                    # Top-k sampling
    "top_p": 0.92,                  # Nucleus sampling threshold
    "do_sample": True,              # Enable sampling (vs greedy)
    "repetition_penalty": 1.1,      # Penalize repeated tokens
}

print("✓ Inference config loaded")
print(f"  Max new tokens: {INFERENCE_CONFIG['max_new_tokens']}")
print(f"  Temperature: {INFERENCE_CONFIG['temperature']}")
print(f"  Sampling: top_k={INFERENCE_CONFIG['top_k']}, top_p={INFERENCE_CONFIG['top_p']}")

---

# SECTION 1: ENVIRONMENT SETUP

> Install dependencies and verify GPU availability

In [ ]:
# ============================================================================
# SECTION 1.1: INSTALL DEPENDENCIES
# ============================================================================
"""
Install required packages. PyTorch is pre-installed in Colab.
Using -q for quiet installation to reduce output noise.
"""

!pip install -q transformers tokenizers datasets matplotlib seaborn

print("✓ Dependencies installed")

In [ ]:
# ============================================================================
# SECTION 1.2: IMPORT LIBRARIES
# ============================================================================
"""
Import all required libraries for tokenizer training, model building, and visualization.
"""

import os
import json
from pathlib import Path

# PyTorch (pre-installed in Colab)
import torch
from torch.utils.data import Dataset, DataLoader

# Hugging Face ecosystem
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Visualization (constitution-mandated)
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("✓ All libraries imported successfully")
print(f"  PyTorch version: {torch.__version__}")
print(f"  Transformers imported")

In [ ]:
# ============================================================================
# SECTION 1.3: GPU AVAILABILITY CHECK
# ============================================================================
"""
Verify GPU availability and print device information.
This notebook is optimized for A100 GPU (40GB VRAM).
"""

if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print("✓ GPU Available!")
    print(f"  Device: {gpu_name}")
    print(f"  VRAM: {gpu_memory:.1f} GB")
    
    # Check if A100
    if "A100" in gpu_name:
        print("  ✓ A100 GPU detected - optimal configuration")
    else:
        print(f"  ⚠️ Warning: Expected A100, got {gpu_name}")
        print("    Training may be slower or require batch size adjustment")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected - training will be very slow")
    print("  Recommendation: Enable GPU in Runtime > Change runtime type > A100")

print(f"\n  Using device: {device}")

---

# SECTION 2: DATA LOADING

> Load and explore the recipe dataset

In [ ]:
# ============================================================================
# SECTION 2.1: LOAD RECIPE TEXT FILE
# ============================================================================
"""
Load the recipe dataset from a plain text file.
Expected format: One recipe per line with [BOS]/[EOS] markers.
"""

# Check if file exists
if not os.path.exists(RECIPE_FILE_PATH):
    raise FileNotFoundError(
        f"Recipe file not found: {RECIPE_FILE_PATH}\n"
        "Please upload your recipe file to the Colab runtime and update RECIPE_FILE_PATH."
    )

# Load recipes
with open(RECIPE_FILE_PATH, "r", encoding="utf-8") as f:
    recipes = [line.strip() for line in f if line.strip()]

print(f"✓ Loaded {len(recipes):,} recipes from {RECIPE_FILE_PATH}")
print(f"\n📄 Sample recipe (first entry):")
print("-" * 60)
print(recipes[0][:500] + "..." if len(recipes[0]) > 500 else recipes[0])
print("-" * 60)

In [ ]:
# ============================================================================
# SECTION 2.2: DATA EXPLORATION & STATISTICS
# ============================================================================
"""
Compute and display statistics about the recipe dataset.
"""

import numpy as np

# Calculate recipe lengths (in characters)
recipe_lengths = [len(recipe) for recipe in recipes]

print("📊 Dataset Statistics")
print("=" * 40)
print(f"  Total recipes: {len(recipes):,}")
print(f"  Min length: {min(recipe_lengths):,} chars")
print(f"  Max length: {max(recipe_lengths):,} chars")
print(f"  Mean length: {np.mean(recipe_lengths):,.1f} chars")
print(f"  Median length: {np.median(recipe_lengths):,.1f} chars")
print(f"  Std deviation: {np.std(recipe_lengths):,.1f} chars")

# Check for [BOS] and [EOS] markers
bos_count = sum(1 for r in recipes if "[BOS]" in r)
eos_count = sum(1 for r in recipes if "[EOS]" in r)
print(f"\n📌 Special Token Coverage")
print(f"  Recipes with [BOS]: {bos_count:,} ({100*bos_count/len(recipes):.1f}%)")
print(f"  Recipes with [EOS]: {eos_count:,} ({100*eos_count/len(recipes):.1f}%)")

In [ ]:
# ============================================================================
# SECTION 2.3: VISUALIZE RECIPE LENGTH DISTRIBUTION
# ============================================================================
"""
Plot the distribution of recipe lengths using seaborn histogram.
"""

fig, ax = plt.subplots(figsize=(12, 6))

sns.histplot(recipe_lengths, bins=50, kde=True, ax=ax, color="steelblue")

ax.set_xlabel("Recipe Length (characters)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Distribution of Recipe Lengths in Dataset", fontsize=14)

# Add vertical line for max sequence length (in chars, approximating 3000 tokens)
# Rough estimate: 1 token ≈ 4 characters
approx_max_chars = MODEL_CONFIG["n_positions"] * 4
ax.axvline(x=approx_max_chars, color="red", linestyle="--", linewidth=2, 
           label=f"Max seq length (~{approx_max_chars:,} chars)")
ax.legend()

plt.tight_layout()
plt.show()

# Count recipes that will be truncated
truncated_count = sum(1 for l in recipe_lengths if l > approx_max_chars)
print(f"\n⚠️ Recipes exceeding max length: {truncated_count:,} ({100*truncated_count/len(recipes):.1f}%)")
print("   These will be truncated during tokenization.")